# Design a Python module that interacts with a RESTful API to perform CRUD operations. The module should include functions for creating, reading, updating, and deleting resources. Describe the structure of the module and how you would handle errors and authentication.

## 📂 Suggested Module Structure
```
api_client/
│
├── __init__.py
├── client.py        # Core API client (CRUD functions)
├── exceptions.py    # Custom error handling
└── config.py        # Authentication / base URL settings
```

---

## ✅ Example Implementation

### `config.py`
```python
BASE_URL = "https://jsonplaceholder.typicode.com"  # Dummy API
API_KEY = None  # Replace with real key if needed
```

---

### `exceptions.py`
```python
class APIError(Exception):
    """Base class for API errors."""
    pass

class AuthenticationError(APIError):
    """Raised when authentication fails."""
    pass

class NotFoundError(APIError):
    """Raised when resource is not found."""
    pass
```

---

### `client.py`
```python
import requests
from .config import BASE_URL, API_KEY
from .exceptions import APIError, AuthenticationError, NotFoundError

class APIClient:
    def __init__(self, base_url=BASE_URL, api_key=API_KEY):
        self.base_url = base_url
        self.headers = {"Authorization": f"Bearer {api_key}"} if api_key else {}

    def create(self, resource, data):
        """POST - Create a new resource"""
        response = requests.post(f"{self.base_url}/{resource}", json=data, headers=self.headers)
        return self._handle_response(response)

    def read(self, resource, resource_id=None):
        """GET - Read resource(s)"""
        url = f"{self.base_url}/{resource}"
        if resource_id:
            url += f"/{resource_id}"
        response = requests.get(url, headers=self.headers)
        return self._handle_response(response)

    def update(self, resource, resource_id, data):
        """PUT - Update an existing resource"""
        response = requests.put(f"{self.base_url}/{resource}/{resource_id}", json=data, headers=self.headers)
        return self._handle_response(response)

    def delete(self, resource, resource_id):
        """DELETE - Remove a resource"""
        response = requests.delete(f"{self.base_url}/{resource}/{resource_id}", headers=self.headers)
        return self._handle_response(response)

    def _handle_response(self, response):
        """Centralized error handling"""
        if response.status_code == 401:
            raise AuthenticationError("Invalid or missing API key.")
        elif response.status_code == 404:
            raise NotFoundError("Resource not found.")
        elif not response.ok:
            raise APIError(f"API error: {response.status_code} - {response.text}")
        return response.json() if response.text else {}
```

---

## 🎯 Usage Example
```python
from api_client.client import APIClient

client = APIClient()

# Create
new_post = client.create("posts", {"title": "Hello", "body": "World", "userId": 1})
print(new_post)

# Read
post = client.read("posts", 1)
print(post)

# Update
updated_post = client.update("posts", 1, {"title": "Updated Title"})
print(updated_post)

# Delete
client.delete("posts", 1)
print("Post deleted")
```

---

## ⚡ Key Design Choices
- **Error Handling**: Centralized in `_handle_response`, raising custom exceptions for clarity.  
- **Authentication**: Uses headers (`Authorization: Bearer <token>`). If no API key is needed, headers remain empty.  
- **Modularity**: Separate files for config, exceptions, and client logic. Easy to extend.  
- **Reusability**: Works with any RESTful API by changing `BASE_URL` and `API_KEY`.  

---